In [1]:

import sys
import os
import platform

print("ShadowFox AI Engineer Internship")
print("Task 3: Advanced RAG Assistant")
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Environment ready!")


ShadowFox AI Engineer Internship
Task 3: Advanced RAG Assistant
Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Environment ready!


In [2]:

%pip install -q fastapi "uvicorn[standard]" streamlit pydantic pypdf sentence-transformers faiss-cpu python-multipart openai httpx pytest


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 73.3 MB/s eta 0:00:00


In [3]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

folders = [
    "api",
    "ui",
    "tests",
    "sample_documents",
    "screenshots",
]

for folder in folders:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT)
print("\nFolders created:")
for folder in folders:
    print("-", folder)

print("\nFolder setup complete!")


Project folder: /content/Task-3-Advanced-RAG-Assistant

Folders created:
- api
- ui
- tests
- sample_documents
- screenshots

Folder setup complete!


In [4]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

files = {
    "requirements.txt": """fastapi
uvicorn[standard]
streamlit
pydantic
pypdf
sentence-transformers
faiss-cpu
python-multipart
openai
httpx
pytest
""",
    ".env.example": """OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
""",
    ".gitignore": """.env
__pycache__/
*.py[cod]
.pytest_cache/
.venv/
venv/
.streamlit/secrets.toml
.DS_Store
""",
    "api/__init__.py": "",
}

for name, content in files.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print("Created:", name)

print("\nBasic project files created!")


Created: requirements.txt
Created: .env.example
Created: .gitignore
Created: api/__init__.py

Basic project files created!


In [5]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

rag_code = r'''
from pathlib import Path
from pypdf import PdfReader

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}

def extract_text(file_path: str) -> str:
    path = Path(file_path)

    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError("Only PDF, TXT, and Markdown files are supported.")

    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    else:
        text = path.read_text(encoding="utf-8", errors="replace")

    if not text.strip():
        raise ValueError("No extractable text found in this document.")

    return text


def chunk_text(text: str, chunk_size: int = 800,
               overlap: int = 120) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    text = " ".join(text.split())
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks
'''

(PROJECT / "api" / "rag.py").write_text(rag_code, encoding="utf-8")

print("Created api/rag.py")
print("Includes PDF/TXT/Markdown extraction and chunking.")


Created api/rag.py
Includes PDF/TXT/Markdown extraction and chunking.


In [6]:

from pathlib import Path
import sys

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
sys.path.insert(0, str(PROJECT))

from api.rag import extract_text, chunk_text

sample = PROJECT / "sample_documents" / "sample.txt"

sample.write_text(
    """Project: PaperPilot
PaperPilot is a fictional document question-answering project.
The example chunk size is 800 characters.
The example overlap is 120 characters.
Embeddings represent text as numerical vectors.
Vector search retrieves text with similar meaning.
The system should not invent answers absent from the documents.
""",
    encoding="utf-8"
)

text = extract_text(str(sample))
chunks = chunk_text(text, chunk_size=100, overlap=20)

print("Document extraction successful!")
print("Extracted characters:", len(text))
print("Number of chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])


Document extraction successful!
Extracted characters: 327
Number of chunks: 4

First chunk:
Project: PaperPilot PaperPilot is a fictional document question-answering project. The example chunk


In [7]:

from pathlib import Path
import sys

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
sys.path.insert(0, str(PROJECT))

vector_code = r'''
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

class VectorStore:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.chunks = []

    def build(self, chunks):
        if not chunks:
            raise ValueError("Cannot index an empty document.")

        self.chunks = list(chunks)
        embeddings = self.model.encode(
            self.chunks,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)
        return len(self.chunks)

    def search(self, question, top_k=3):
        if self.index is None:
            raise RuntimeError("Build the index before searching.")

        query_vector = self.model.encode(
            [question],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        k = min(top_k, len(self.chunks))
        scores, indices = self.index.search(query_vector, k)

        return [
            {"text": self.chunks[int(i)], "score": float(score)}
            for score, i in zip(scores[0], indices[0])
            if i >= 0
        ]
'''

(PROJECT / "api" / "vector_store.py").write_text(
    vector_code, encoding="utf-8"
)

from api.rag import extract_text, chunk_text
from api.vector_store import VectorStore

sample = PROJECT / "sample_documents" / "sample.txt"
document_text = extract_text(str(sample))
document_chunks = chunk_text(document_text, chunk_size=800, overlap=120)

store = VectorStore()
count = store.build(document_chunks)

print("Vector index created successfully!")
print("Indexed chunks:", count)

results = store.search("What is the example chunk size?", top_k=3)

print("\nRetrieved results:")
for i, result in enumerate(results, 1):
    print(f"\nResult {i} | Score: {result['score']:.3f}")
    print(result["text"])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index created successfully!
Indexed chunks: 1

Retrieved results:

Result 1 | Score: 0.434
Project: PaperPilot PaperPilot is a fictional document question-answering project. The example chunk size is 800 characters. The example overlap is 120 characters. Embeddings represent text as numerical vectors. Vector search retrieves text with similar meaning. The system should not invent answers absent from the documents.


In [8]:

from pathlib import Path

api_dir = PROJECT / "api"
api_dir.mkdir(parents=True, exist_ok=True)
(api_dir / "__init__.py").touch()

qa_code = r'''
from transformers import pipeline

class GroundedQA:
    def __init__(self, model_name="google/flan-t5-small"):
        self.generator = pipeline(
            "text2text-generation",
            model=model_name
        )

    def answer(self, question, vector_store, top_k=3):
        results = vector_store.search(question, top_k=top_k)

        if not results:
            return {
                "answer": "The answer was not found in the uploaded documents.",
                "sources": []
            }

        context = "\n\n".join(
            item["text"] for item in results
        )

        prompt = f"""
Answer the question using only the supplied context.
Do not add facts from outside the context.
If the context does not contain the answer, say:
The answer was not found in the uploaded documents.

Context:
{context}

Question: {question}
Answer:
"""

        generated = self.generator(
            prompt,
            max_new_tokens=120,
            do_sample=False
        )[0]["generated_text"].strip()

        return {
            "answer": generated,
            "sources": results
        }
'''

(api_dir / "qa.py").write_text(qa_code, encoding="utf-8")
print("api/qa.py created successfully!")


api/qa.py created successfully!


In [9]:

%pip install -q transformers sentencepiece


In [11]:

from pathlib import Path

qa_code = r'''
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class GroundedQA:
    def __init__(self, model_name="google/flan-t5-small"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.model.eval()

    def answer(self, question, vector_store, top_k=3):
        results = vector_store.search(question, top_k=top_k)

        if not results:
            return {
                "answer": "The answer was not found in the uploaded documents.",
                "sources": []
            }

        context = "\n\n".join(item["text"] for item in results)

        prompt = (
            "Answer using only the context below. "
            "If the answer is absent, say that it was not found "
            "in the uploaded documents.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\nAnswer:"
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False
            )

        generated = self.tokenizer.decode(
            output[0], skip_special_tokens=True
        ).strip()

        return {
            "answer": generated,
            "sources": results
        }
'''

qa_file = PROJECT / "api" / "qa.py"
qa_file.parent.mkdir(parents=True, exist_ok=True)
qa_file.write_text(qa_code, encoding="utf-8")

print("Updated api/qa.py successfully!")


Updated api/qa.py successfully!


In [12]:

import importlib
import api.qa

importlib.reload(api.qa)

from api.qa import GroundedQA

qa = GroundedQA()

question = "What is the example chunk size?"
response = qa.answer(question, store, top_k=3)

print("Question:", question)
print("Answer:", response["answer"])
print("\nRetrieved sources:")

for source in response["sources"]:
    print("Score:", round(source["score"], 3))
    print("Text:", source["text"][:300])


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What is the example chunk size?
Answer: 800 characters

Retrieved sources:
Score: 0.434
Text: Project: PaperPilot PaperPilot is a fictional document question-answering project. The example chunk size is 800 characters. The example overlap is 120 characters. Embeddings represent text as numerical vectors. Vector search retrieves text with similar meaning. The system should not invent answers 


In [13]:

from pathlib import Path

ingestion_code = r'''
from pathlib import Path
from pypdf import PdfReader

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}
MAX_FILE_SIZE_MB = 20

def extract_text(file_path):
    path = Path(file_path)

    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError("Only PDF, TXT and Markdown files are supported.")

    if path.stat().st_size > MAX_FILE_SIZE_MB * 1024 * 1024:
        raise ValueError("File exceeds the 20 MB size limit.")

    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))

        if reader.is_encrypted:
            raise ValueError("Encrypted PDFs are not supported.")

        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    else:
        text = path.read_text(encoding="utf-8", errors="replace")

    text = "\n".join(line.strip() for line in text.splitlines())
    text = text.strip()

    if not text:
        raise ValueError("No readable text found in the document.")

    return text


def chunk_text(text, chunk_size=800, overlap=120):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be between 0 and chunk_size.")

    text = text.strip()
    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks
'''

ingestion_file = PROJECT / "api" / "ingestion.py"
ingestion_file.parent.mkdir(parents=True, exist_ok=True)
ingestion_file.write_text(ingestion_code, encoding="utf-8")

print("Created:", ingestion_file)
print("Document ingestion module ready!")


Created: /content/Task-3-Advanced-RAG-Assistant/api/ingestion.py
Document ingestion module ready!


In [14]:

from api.ingestion import extract_text, chunk_text
from pathlib import Path

sample_dir = PROJECT / "sample_documents"
sample_dir.mkdir(parents=True, exist_ok=True)

sample_file = sample_dir / "sample.txt"

if not sample_file.exists():
    sample_file.write_text(
        "PaperPilot is a fictional document question-answering project. "
        "The example chunk size is 800 characters and the overlap is "
        "120 characters. Embeddings represent text as vectors.",
        encoding="utf-8"
    )

text = extract_text(sample_file)
chunks = chunk_text(text, chunk_size=800, overlap=120)

print("Extraction successful:", bool(text))
print("Characters extracted:", len(text))
print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0][:300])


Extraction successful: True
Characters extracted: 326
Number of chunks: 1
First chunk: Project: PaperPilot
PaperPilot is a fictional document question-answering project.
The example chunk size is 800 characters.
The example overlap is 120 characters.
Embeddings represent text as numerical vectors.
Vector search retrieves text with similar meaning.
The system should not invent answers 


In [15]:

from api.vector_store import VectorStore
from api.qa import GroundedQA
from api.ingestion import extract_text, chunk_text

document_path = PROJECT / "sample_documents" / "sample.txt"

document_text = extract_text(document_path)
document_chunks = chunk_text(
    document_text,
    chunk_size=800,
    overlap=120
)

test_store = VectorStore()
test_store.build(document_chunks)

test_qa = GroundedQA()

question = "What is the example chunk size?"
result = test_qa.answer(question, test_store, top_k=3)

print("Question:", question)
print("Answer:", result["answer"])
print("Sources returned:", len(result["sources"]))

for i, source in enumerate(result["sources"], 1):
    print(f"\nSource {i} | Score: {source['score']:.3f}")
    print(source["text"][:300])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What is the example chunk size?
Answer: 800 characters
Sources returned: 1

Source 1 | Score: 0.434
Project: PaperPilot
PaperPilot is a fictional document question-answering project.
The example chunk size is 800 characters.
The example overlap is 120 characters.
Embeddings represent text as numerical vectors.
Vector search retrieves text with similar meaning.
The system should not invent answers 


In [16]:

%pip install -q fastapi uvicorn python-multipart httpx


In [17]:

from pathlib import Path

main_code = r'''
from pathlib import Path
from tempfile import NamedTemporaryFile
from threading import Lock

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field

from api.ingestion import extract_text, chunk_text
from api.vector_store import VectorStore
from api.qa import GroundedQA

app = FastAPI(
    title="Advanced RAG Assistant API",
    version="1.0.0",
    description="Document-grounded question answering."
)

store = VectorStore()
qa_model = None
document_loaded = False
qa_lock = Lock()

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=1000)

@app.get("/")
def health():
    return {
        "status": "running",
        "service": "Advanced RAG Assistant"
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    global document_loaded

    suffix = Path(file.filename or "").suffix.lower()
    if suffix not in {".pdf", ".txt", ".md"}:
        raise HTTPException(
            status_code=400,
            detail="Only PDF, TXT and Markdown files are supported."
        )

    data = await file.read()

    if not data:
        raise HTTPException(status_code=400, detail="Uploaded file is empty.")

    if len(data) > 20 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Maximum file size is 20 MB.")

    temp_path = None
    try:
        with NamedTemporaryFile(
            suffix=suffix, delete=False
        ) as temp:
            temp.write(data)
            temp_path = Path(temp.name)

        text = extract_text(temp_path)
        chunks = chunk_text(text, chunk_size=800, overlap=120)

        if not chunks:
            raise HTTPException(
                status_code=422,
                detail="No readable text found in the document."
            )

        store.build(chunks)
        document_loaded = True

        return {
            "filename": Path(file.filename).name,
            "characters_extracted": len(text),
            "chunks_indexed": len(chunks),
            "message": "Document uploaded and indexed successfully."
        }

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=422, detail=str(exc))
    finally:
        if temp_path and temp_path.exists():
            temp_path.unlink(missing_ok=True)
        await file.close()

@app.post("/ask")
def ask_question(request: QuestionRequest):
    global qa_model

    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be blank.")

    if not document_loaded:
        raise HTTPException(
            status_code=400,
            detail="Upload a document before asking questions."
        )

    try:
        with qa_lock:
            if qa_model is None:
                qa_model = GroundedQA()

            result = qa_model.answer(
                request.question.strip(), store, top_k=3
            )

        return {
            "question": request.question,
            "answer": result["answer"],
            "sources": result["sources"]
        }

    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))
'''

main_file = PROJECT / "api" / "main.py"
main_file.write_text(main_code, encoding="utf-8")

print("Created:", main_file)
print("FastAPI backend code is ready.")


Created: /content/Task-3-Advanced-RAG-Assistant/api/main.py
FastAPI backend code is ready.


In [18]:

from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

health_response = client.get("/")
print("Health status:", health_response.status_code)
print("Health response:", health_response.json())

sample_path = PROJECT / "sample_documents" / "sample.txt"

with open(sample_path, "rb") as f:
    upload_response = client.post(
        "/upload",
        files={
            "file": (
                "sample.txt",
                f,
                "text/plain"
            )
        }
    )

print("\nUpload status:", upload_response.status_code)
print("Upload response:", upload_response.json())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Health status: 200
Health response: {'status': 'running', 'service': 'Advanced RAG Assistant'}

Upload status: 200
Upload response: {'filename': 'sample.txt', 'characters_extracted': 326, 'chunks_indexed': 1, 'message': 'Document uploaded and indexed successfully.'}


In [19]:

ask_response = client.post(
    "/ask",
    json={"question": "What is the example chunk size?"}
)

print("Ask status:", ask_response.status_code)
print("Response:", ask_response.json())


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Ask status: 200
Response: {'question': 'What is the example chunk size?', 'answer': '800 characters', 'sources': [{'text': 'Project: PaperPilot\nPaperPilot is a fictional document question-answering project.\nThe example chunk size is 800 characters.\nThe example overlap is 120 characters.\nEmbeddings represent text as numerical vectors.\nVector search retrieves text with similar meaning.\nThe system should not invent answers absent from the documents.', 'score': 0.4340517222881317}]}


In [20]:

from pathlib import Path

ui_code = r'''
import streamlit as st
import requests

st.set_page_config(
    page_title="Advanced RAG Assistant",
    page_icon="📚",
    layout="wide"
)

st.title("Advanced RAG Document Assistant")
st.caption("Upload a document and ask questions grounded in its content.")

API_URL = "http://127.0.0.1:8000"

uploaded_file = st.file_uploader(
    "Upload a document",
    type=["pdf", "txt", "md"]
)

if uploaded_file is not None:
    if st.button("Upload and index document"):
        try:
            response = requests.post(
                f"{API_URL}/upload",
                files={
                    "file": (
                        uploaded_file.name,
                        uploaded_file.getvalue(),
                        uploaded_file.type or "application/octet-stream"
                    )
                },
                timeout=180
            )
            response.raise_for_status()
            st.session_state["document_ready"] = True
            st.success("Document indexed successfully.")
            st.json(response.json())
        except requests.RequestException as exc:
            st.error(f"Upload failed: {exc}")

question = st.text_input(
    "Ask a question",
    max_chars=1000,
    placeholder="What does the document say?"
)

if st.button("Submit question"):
    if not st.session_state.get("document_ready", False):
        st.warning("Upload and index a document first.")
    elif not question.strip():
        st.warning("Please enter a question.")
    else:
        try:
            with st.spinner("Searching the document..."):
                response = requests.post(
                    f"{API_URL}/ask",
                    json={"question": question.strip()},
                    timeout=180
                )
                response.raise_for_status()
                result = response.json()

            st.subheader("Answer")
            st.write(result["answer"])

            st.subheader("Retrieved sources")
            for index, source in enumerate(result.get("sources", []), 1):
                with st.expander(f"Source {index}"):
                    st.write(source.get("text", ""))
                    if "score" in source:
                        st.write(f"Similarity score: {source['score']:.3f}")

        except requests.RequestException as exc:
            st.error(f"Question request failed: {exc}")
'''

ui_path = PROJECT / "api" / "ui.py"
ui_path.write_text(ui_code, encoding="utf-8")

print("Created:", ui_path)


Created: /content/Task-3-Advanced-RAG-Assistant/api/ui.py


In [21]:

import py_compile

py_compile.compile(
    str(PROJECT / "api" / "ui.py"),
    doraise=True
)

print("UI syntax check passed!")


UI syntax check passed!


In [22]:

from pathlib import Path

required_files = [
    "api/__init__.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py",
    "requirements.txt",
    "README.md",
]

for name in required_files:
    path = PROJECT / name
    print(("OK      " if path.exists() else "MISSING "), name)


OK       api/__init__.py
OK       api/ingestion.py
OK       api/vector_store.py
OK       api/qa.py
OK       api/main.py
OK       api/ui.py
OK       requirements.txt
MISSING  README.md


In [23]:

init_file = PROJECT / "api" / "__init__.py"
init_file.parent.mkdir(parents=True, exist_ok=True)
init_file.touch(exist_ok=True)
print("Package initialization file ready.")


Package initialization file ready.


In [24]:

readme_text = """# ShadowFox AI Engineer Internship
## Task 3: Advanced RAG Document Assistant

A document question-answering application designed to retrieve relevant
information from uploaded PDF, TXT, and Markdown files.

### Planned Features
- Multi-format document ingestion
- Text extraction and chunking
- Embedding generation and vector search
- Document-grounded question answering
- Retrieved source context and relevance scores
- FastAPI backend and Streamlit user interface
- Input validation and reproducible setup

### Project Structure
```text
Task-3-Advanced-RAG-Assistant/
├── api/
│   ├── __init__.py
│   ├── ingestion.py
│   ├── vector_store.py
│   ├── qa.py
│   ├── main.py
│   └── ui.py
├── requirements.txt
├── README.md
├── Dockerfile
└── tests/
```

### Installation
```bash
pip install -r requirements.txt
```

### Running the Application
Start the FastAPI backend:
```bash
uvicorn api.main:app --reload
```

Start the Streamlit interface in a separate terminal:
```bash
streamlit run api/ui.py
```

### Testing
Test document upload, retrieval, question answering, source display,
invalid file handling, and questions unsupported by the document.

### Notes
The actual implemented features and configuration should be verified
before deployment. Localhost URLs require a running backend.
"""

readme_path = PROJECT / "README.md"
readme_path.write_text(readme_text, encoding="utf-8")

print("README created:", readme_path)


README created: /content/Task-3-Advanced-RAG-Assistant/README.md


In [25]:

required_files = [
    "api/__init__.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py",
    "requirements.txt",
    "README.md",
]

for name in required_files:
    path = PROJECT / name
    print(f"{'OK' if path.is_file() else 'MISSING'}: {name}")


OK: api/__init__.py
OK: api/ingestion.py
OK: api/vector_store.py
OK: api/qa.py
OK: api/main.py
OK: api/ui.py
OK: requirements.txt
OK: README.md


In [26]:

main_file = PROJECT / "api" / "main.py"
print(main_file.read_text(encoding="utf-8"))



from pathlib import Path
from tempfile import NamedTemporaryFile
from threading import Lock

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field

from api.ingestion import extract_text, chunk_text
from api.vector_store import VectorStore
from api.qa import GroundedQA

app = FastAPI(
    title="Advanced RAG Assistant API",
    version="1.0.0",
    description="Document-grounded question answering."
)

store = VectorStore()
qa_model = None
document_loaded = False
qa_lock = Lock()

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=1000)

@app.get("/")
def health():
    return {
        "status": "running",
        "service": "Advanced RAG Assistant"
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    global document_loaded

    suffix = Path(file.filename or "").suffix.lower()
    if suffix not in {".pdf", ".txt", ".md"}:
        raise HTTPException(
            status_c

In [27]:

import py_compile

for filename in [
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py",
]:
    py_compile.compile(
        str(PROJECT / filename),
        doraise=True
    )
    print("Syntax OK:", filename)


Syntax OK: api/ingestion.py
Syntax OK: api/vector_store.py
Syntax OK: api/qa.py
Syntax OK: api/main.py
Syntax OK: api/ui.py


In [28]:

main_file = PROJECT / "api" / "main.py"
print(main_file.read_text(encoding="utf-8"))



from pathlib import Path
from tempfile import NamedTemporaryFile
from threading import Lock

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field

from api.ingestion import extract_text, chunk_text
from api.vector_store import VectorStore
from api.qa import GroundedQA

app = FastAPI(
    title="Advanced RAG Assistant API",
    version="1.0.0",
    description="Document-grounded question answering."
)

store = VectorStore()
qa_model = None
document_loaded = False
qa_lock = Lock()

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=1000)

@app.get("/")
def health():
    return {
        "status": "running",
        "service": "Advanced RAG Assistant"
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    global document_loaded

    suffix = Path(file.filename or "").suffix.lower()
    if suffix not in {".pdf", ".txt", ".md"}:
        raise HTTPException(
            status_c

In [29]:

import ast

main_file = PROJECT / "api" / "main.py"
tree = ast.parse(main_file.read_text(encoding="utf-8"))

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        for decorator in node.decorator_list:
            if isinstance(decorator, ast.Call):
                func = decorator.func
                if isinstance(func, ast.Attribute):
                    if func.attr in ("get", "post", "put", "delete"):
                        route = (
                            ast.unparse(decorator.args[0])
                            if decorator.args else "?"
                        )
                        print(func.attr.upper(), route, "->", node.name)


GET '/' -> health
POST '/upload' -> upload_document
POST '/ask' -> ask_question


In [30]:

req_file = PROJECT / "requirements.txt"
print(req_file.read_text(encoding="utf-8"))


fastapi
uvicorn[standard]
streamlit
pydantic
pypdf
sentence-transformers
faiss-cpu
python-multipart
openai
httpx
pytest



In [31]:

import subprocess
import sys

api_process = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "api.main:app",
    "--host", "127.0.0.1",
    "--port", "8000"
], cwd=str(PROJECT))

print("API process started:", api_process.pid)


API process started: 6135


In [32]:

import time
import urllib.request

time.sleep(3)

try:
    with urllib.request.urlopen(
        "http://127.0.0.1:8000/openapi.json",
        timeout=10
    ) as response:
        print("Status:", response.status)
        print(response.read().decode("utf-8")[:2500])
except Exception as exc:
    print("API check failed:", repr(exc))


Status: 200
{"openapi":"3.1.0","info":{"title":"Advanced RAG Assistant API","description":"Document-grounded question answering.","version":"1.0.0"},"paths":{"/":{"get":{"summary":"Health","operationId":"health__get","responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}}}}},"/upload":{"post":{"summary":"Upload Document","operationId":"upload_document_upload_post","requestBody":{"content":{"multipart/form-data":{"schema":{"$ref":"#/components/schemas/Body_upload_document_upload_post"}}},"required":true},"responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}},"422":{"description":"Validation Error","content":{"application/json":{"schema":{"$ref":"#/components/schemas/HTTPValidationError"}}}}}}},"/ask":{"post":{"summary":"Ask Question","operationId":"ask_question_ask_post","requestBody":{"content":{"application/json":{"schema":{"$ref":"#/components/schemas/QuestionRequest"}}},"required":true},"res

In [33]:

import json
import urllib.request

url = "http://127.0.0.1:8000/openapi.json"

with urllib.request.urlopen(url, timeout=15) as response:
    schema = json.loads(response.read().decode("utf-8"))

print("API:", schema.get("info", {}).get("title"))
print("\nAvailable endpoints:")

for path, methods in schema.get("paths", {}).items():
    for method, details in methods.items():
        print(
            method.upper(),
            path,
            "-",
            details.get("summary", "")
        )


API: Advanced RAG Assistant API

Available endpoints:
GET / - Health
POST /upload - Upload Document
POST /ask - Ask Question


In [34]:

import urllib.request
import json

try:
    with urllib.request.urlopen(
        "http://127.0.0.1:8000/",
        timeout=10
    ) as response:
        print("Root status:", response.status)
        print(response.read().decode("utf-8")[:1000])
except Exception as exc:
    print("Root endpoint:", exc)


Root status: 200
{"status":"running","service":"Advanced RAG Assistant"}


In [35]:

for path, methods in schema.get("paths", {}).items():
    for method, details in methods.items():
        if any(word in path.lower() for word in ["upload", "ask", "query"]):
            print("\n", method.upper(), path)
            print("Summary:", details.get("summary"))
            print("Parameters:", details.get("parameters", []))
            print("Request body:")
            print(json.dumps(
                details.get("requestBody", {}),
                indent=2
            )[:3000])
            print("Responses:")
            print(json.dumps(
                details.get("responses", {}),
                indent=2
            )[:2000])



 POST /upload
Summary: Upload Document
Parameters: []
Request body:
{
  "content": {
    "multipart/form-data": {
      "schema": {
        "$ref": "#/components/schemas/Body_upload_document_upload_post"
      }
    }
  },
  "required": true
}
Responses:
{
  "200": {
    "description": "Successful Response",
    "content": {
      "application/json": {
        "schema": {}
      }
    }
  },
  "422": {
    "description": "Validation Error",
    "content": {
      "application/json": {
        "schema": {
          "$ref": "#/components/schemas/HTTPValidationError"
        }
      }
    }
  }
}

 POST /ask
Summary: Ask Question
Parameters: []
Request body:
{
  "content": {
    "application/json": {
      "schema": {
        "$ref": "#/components/schemas/QuestionRequest"
      }
    }
  },
  "required": true
}
Responses:
{
  "200": {
    "description": "Successful Response",
    "content": {
      "application/json": {
        "schema": {}
      }
    }
  },
  "422": {
    "description"

In [36]:

schema = json.loads(
    urllib.request.urlopen(
        "http://127.0.0.1:8000/openapi.json"
    ).read().decode("utf-8")
)

print(json.dumps(
    schema["components"]["schemas"].get(
        "Body_upload_document_upload_post", {}
    ),
    indent=2
))


{
  "properties": {
    "file": {
      "type": "string",
      "contentMediaType": "application/octet-stream",
      "title": "File"
    }
  },
  "type": "object",
  "required": [
    "file"
  ],
  "title": "Body_upload_document_upload_post"
}


In [37]:

from google.colab import files
from pathlib import Path

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


Saving ShadowFox_Sample_Document.pdf to ShadowFox_Sample_Document.pdf
Uploaded files: ['ShadowFox_Sample_Document.pdf']


In [39]:

import os

print(os.path.exists("/content/ShadowFox_Sample_Document.pdf"))
print(os.listdir("/content")[:30])


True
['.config', 'Task-3-Advanced-RAG-Assistant', 'ShadowFox_Sample_Document.pdf', 'sample_data']


In [40]:

from pathlib import Path
import requests

pdf_path = "/content/ShadowFox_Sample_Document.pdf"

if not Path(pdf_path).exists():
    raise FileNotFoundError(
        f"PDF not found: {pdf_path}"
    )

with open(pdf_path, "rb") as f:
    response = requests.post(
        "http://127.0.0.1:8000/upload",
        files={
            "file": (
                Path(pdf_path).name,
                f,
                "application/pdf"
            )
        },
        timeout=120
    )

print("Status:", response.status_code)
print("Response:", response.text[:3000])


Status: 200
Response: {"filename":"ShadowFox_Sample_Document.pdf","characters_extracted":1964,"chunks_indexed":3,"message":"Document uploaded and indexed successfully."}


In [41]:

import requests
import json

url = "http://127.0.0.1:8000/ask"

payload = {
    "question": "What is the example chunk size?"
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))


Status: 200
{
  "question": "What is the example chunk size?",
  "answer": "800 characters",
  "sources": [
    {
      "text": "in the documents.\n5. Example Facts for Testing\nThe fictional test project is named PaperPilot. It uses a chunk size of 800 characters and an overlap of 120\ncharacters for its initial experiment. The embedding model used in this example is all-MiniLM-L6-v2. These\nsettings are illustrative, not universal best practices.\nSuggested test questions\n\u007f What is machine learning? \u007f What does an embedding represent? \u007f Why does the system split text into chunks? \u007f\nWhat is the fictional project called? \u007f Which chunk size is used in the example? \u007f What should the system do if the\nanswer is not in the document?",
      "score": 0.48562806844711304
    },
    {
      "text": "sed learning, unsupervised learning, and\nreinforcement learning.\n3. Embeddings and Vector Search\nAn embedding represents text as a list of numbers called a vecto

In [42]:

questions = [
    "What is the example chunk overlap?",
    "What do embeddings represent?",
    "How does vector search work?"
]

for question in questions:
    response = requests.post(
        "http://127.0.0.1:8000/ask",
        json={"question": question},
        timeout=120
    )

    print("\nQuestion:", question)
    print("Status:", response.status_code)

    if response.ok:
        print("Answer:", response.json())
    else:
        print("Error:", response.text[:500])



Question: What is the example chunk overlap?
Status: 200
Answer: {'question': 'What is the example chunk overlap?', 'answer': '120 characters', 'sources': [{'text': 'in the documents.\n5. Example Facts for Testing\nThe fictional test project is named PaperPilot. It uses a chunk size of 800 characters and an overlap of 120\ncharacters for its initial experiment. The embedding model used in this example is all-MiniLM-L6-v2. These\nsettings are illustrative, not universal best practices.\nSuggested test questions\n\x7f What is machine learning? \x7f What does an embedding represent? \x7f Why does the system split text into chunks? \x7f\nWhat is the fictional project called? \x7f Which chunk size is used in the example? \x7f What should the system do if the\nanswer is not in the document?', 'score': 0.4672747254371643}, {'text': "sed learning, unsupervised learning, and\nreinforcement learning.\n3. Embeddings and Vector Search\nAn embedding represents text as a list of numbers called a ve

In [43]:

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"question": "What is the example chunk size?"},
    timeout=120
)

if response.ok:
    result = response.json()

    print("Answer:")
    print(result.get("answer", "No answer field"))

    print("\nRetrieved sources:")
    print(json.dumps(
        result.get("sources", []),
        indent=2
    ))
else:
    print("Status:", response.status_code)
    print(response.text)


Answer:
800 characters

Retrieved sources:
[
  {
    "text": "in the documents.\n5. Example Facts for Testing\nThe fictional test project is named PaperPilot. It uses a chunk size of 800 characters and an overlap of 120\ncharacters for its initial experiment. The embedding model used in this example is all-MiniLM-L6-v2. These\nsettings are illustrative, not universal best practices.\nSuggested test questions\n\u007f What is machine learning? \u007f What does an embedding represent? \u007f Why does the system split text into chunks? \u007f\nWhat is the fictional project called? \u007f Which chunk size is used in the example? \u007f What should the system do if the\nanswer is not in the document?",
    "score": 0.48562806844711304
  },
  {
    "text": "sed learning, unsupervised learning, and\nreinforcement learning.\n3. Embeddings and Vector Search\nAn embedding represents text as a list of numbers called a vector. Texts with similar meanings can have vectors\nthat are close in an embed

In [44]:

import requests

API_URL = "http://127.0.0.1:8000/ask"

questions = [
    "What is the example chunk size?",
    "What is the example chunk overlap?",
    "What do embeddings represent?",
    "How does vector search work?",
    "What is the name of the fictional project?"
]

for question in questions:
    print("\n" + "=" * 50)
    print("QUESTION:", question)

    try:
        response = requests.post(
            API_URL,
            json={"question": question},
            timeout=120
        )

        print("STATUS:", response.status_code)

        if response.ok:
            data = response.json()
            print("ANSWER:", data.get("answer", data))
        else:
            print("ERROR:", response.text[:500])

    except Exception as error:
        print("REQUEST ERROR:", error)



QUESTION: What is the example chunk size?
STATUS: 200
ANSWER: 800 characters

QUESTION: What is the example chunk overlap?
STATUS: 200
ANSWER: 120 characters

QUESTION: What do embeddings represent?
STATUS: 200
ANSWER: text as a list of numbers called a vector

QUESTION: How does vector search work?
STATUS: 200
ANSWER: Compares vectors to retrieve document passages relevant to a user's question.

QUESTION: What is the name of the fictional project?
STATUS: 200
ANSWER: PaperPilot


In [45]:

question = "What is the capital of Japan?"

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"question": question},
    timeout=120
)

print("Status:", response.status_code)

if response.ok:
    print(response.json())
else:
    print(response.text[:1000])


Status: 200
{'question': 'What is the capital of Japan?', 'answer': 'Japan', 'sources': [{'text': 'Synthetic practice document for ShadowFox AI Engineer Intermediate project\nPage 1\nSample Document for AI Question Answering\nA short practice PDF for upload, extraction, chunking, embedding, and retrieval tests.\n1. Artificial Intelligence\nArtificial Intelligence (AI) is a field of computer science focused on building systems that perform tasks normally\nassociated with human intelligence. Examples include language understanding, image recognition, planning, and\ndecision support.\n2. Machine Learning\nMachine learning (ML) is a subset of AI. It enables a system to learn patterns from examples rather than relying\nonly on hand-written rules. Common learning types include supervised learning, unsupervised learning, and\nreinforcement learning.\n3. Embeddings and Vector Search\nAn embedding represents', 'score': 0.01769936829805374}, {'text': "sed learning, unsupervised learning, and\nre

In [46]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-AI-Document-QA")

for name in [
    "README.md",
    "requirements.txt",
    "Dockerfile",
    ".gitignore",
    "api/__init__.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py"
]:
    path = PROJECT / name
    print(
        ("OK      " if path.exists() else "MISSING "),
        name
    )


MISSING  README.md
MISSING  requirements.txt
MISSING  Dockerfile
MISSING  .gitignore
MISSING  api/__init__.py
MISSING  api/ingestion.py
MISSING  api/vector_store.py
MISSING  api/qa.py
MISSING  api/main.py
MISSING  api/ui.py


In [47]:

from pathlib import Path

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

for root in search_roots:
    if not root.exists():
        continue

    print(f"\nSearching in: {root}")

    for filename in [
        "ingestion.py",
        "vector_store.py",
        "main.py",
        "README.md",
        "requirements.txt"
    ]:
        matches = list(root.glob(f"**/{filename}"))[:10]

        for match in matches:
            print(filename, "->", match)



Searching in: /content
ingestion.py -> /content/Task-3-Advanced-RAG-Assistant/api/ingestion.py
vector_store.py -> /content/Task-3-Advanced-RAG-Assistant/api/vector_store.py
main.py -> /content/Task-3-Advanced-RAG-Assistant/api/main.py
README.md -> /content/Task-3-Advanced-RAG-Assistant/README.md
README.md -> /content/sample_data/README.md
requirements.txt -> /content/Task-3-Advanced-RAG-Assistant/requirements.txt


In [48]:

from pathlib import Path

PROJECT = Path(
    "/content/drive/MyDrive/Task-3-Advanced-AI-Document-QA"
)

print("Project exists:", PROJECT.exists())
print("Project path:", PROJECT)

if PROJECT.exists():
    for item in sorted(PROJECT.rglob("*")):
        if item.is_file():
            print(item.relative_to(PROJECT))


Project exists: False
Project path: /content/drive/MyDrive/Task-3-Advanced-AI-Document-QA


In [49]:

from pathlib import Path

for root in [Path("/content"), Path("/content/drive")]:
    print("\nChecking:", root)

    if root.exists():
        for name in [
            "main.py",
            "ingestion.py",
            "vector_store.py",
            "qa.py",
            "ui.py",
            "requirements.txt",
            "README.md"
        ]:
            matches = list(root.glob(f"**/{name}"))[:5]

            for path in matches:
                print("FOUND:", path)



Checking: /content
FOUND: /content/Task-3-Advanced-RAG-Assistant/api/main.py
FOUND: /content/Task-3-Advanced-RAG-Assistant/api/ingestion.py
FOUND: /content/Task-3-Advanced-RAG-Assistant/api/vector_store.py
FOUND: /content/Task-3-Advanced-RAG-Assistant/api/qa.py
FOUND: /content/Task-3-Advanced-RAG-Assistant/api/ui.py
FOUND: /content/Task-3-Advanced-RAG-Assistant/requirements.txt
FOUND: /content/Task-3-Advanced-RAG-Assistant/README.md
FOUND: /content/sample_data/README.md

Checking: /content/drive


In [50]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

print("Project exists:", PROJECT.exists())
print("Project path:", PROJECT)

for name in [
    "README.md",
    "requirements.txt",
    "Dockerfile",
    ".gitignore",
    "api/__init__.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py",
]:
    path = PROJECT / name
    print(("OK       " if path.exists() else "MISSING  ") + name)


Project exists: True
Project path: /content/Task-3-Advanced-RAG-Assistant
OK       README.md
OK       requirements.txt
MISSING  Dockerfile
OK       .gitignore
OK       api/__init__.py
OK       api/ingestion.py
OK       api/vector_store.py
OK       api/qa.py
OK       api/main.py
OK       api/ui.py


In [51]:

import py_compile

files = [
    "api/__init__.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/main.py",
    "api/ui.py",
]

for name in files:
    path = PROJECT / name

    if path.exists():
        try:
            py_compile.compile(str(path), doraise=True)
            print("SYNTAX OK:", name)
        except py_compile.PyCompileError as exc:
            print("SYNTAX ERROR:", name)
            print(exc)
    else:
        print("MISSING:", name)


SYNTAX OK: api/__init__.py
SYNTAX OK: api/ingestion.py
SYNTAX OK: api/vector_store.py
SYNTAX OK: api/qa.py
SYNTAX OK: api/main.py
SYNTAX OK: api/ui.py


In [52]:

print((PROJECT / "requirements.txt").read_text())


fastapi
uvicorn[standard]
streamlit
pydantic
pypdf
sentence-transformers
faiss-cpu
python-multipart
openai
httpx
pytest



In [53]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

dockerfile_content = """FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

dockerfile_path = PROJECT / "Dockerfile"
dockerfile_path.write_text(dockerfile_content, encoding="utf-8")

print("Dockerfile created:", dockerfile_path.exists())
print(dockerfile_path.read_text())


Dockerfile created: True
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]



In [54]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

for filename in ["api/main.py", "api/qa.py", "api/ui.py"]:
    path = PROJECT / filename
    print("\n" + "=" * 60)
    print("FILE:", filename)
    print("=" * 60)

    if path.exists():
        lines = path.read_text(encoding="utf-8").splitlines()
        for number, line in enumerate(lines[:100], start=1):
            print(f"{number:03}: {line}")
    else:
        print("FILE NOT FOUND")



FILE: api/main.py
001: 
002: from pathlib import Path
003: from tempfile import NamedTemporaryFile
004: from threading import Lock
005: 
006: from fastapi import FastAPI, UploadFile, File, HTTPException
007: from pydantic import BaseModel, Field
008: 
009: from api.ingestion import extract_text, chunk_text
010: from api.vector_store import VectorStore
011: from api.qa import GroundedQA
012: 
013: app = FastAPI(
014:     title="Advanced RAG Assistant API",
015:     version="1.0.0",
016:     description="Document-grounded question answering."
017: )
018: 
019: store = VectorStore()
020: qa_model = None
021: document_loaded = False
022: qa_lock = Lock()
023: 
024: class QuestionRequest(BaseModel):
025:     question: str = Field(min_length=1, max_length=1000)
026: 
027: @app.get("/")
028: def health():
029:     return {
030:         "status": "running",
031:         "service": "Advanced RAG Assistant"
032:     }
033: 
034: @app.post("/upload")
035: async def upload_document(file: UploadFi

In [55]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
path = PROJECT / "api/main.py"

lines = path.read_text(encoding="utf-8").splitlines()

print("Total lines:", len(lines))

for number, line in enumerate(lines[40:], start=41):
    print(f"{number:03}: {line}")


Total lines: 118
041:             status_code=400,
042:             detail="Only PDF, TXT and Markdown files are supported."
043:         )
044: 
045:     data = await file.read()
046: 
047:     if not data:
048:         raise HTTPException(status_code=400, detail="Uploaded file is empty.")
049: 
050:     if len(data) > 20 * 1024 * 1024:
051:         raise HTTPException(status_code=413, detail="Maximum file size is 20 MB.")
052: 
053:     temp_path = None
054:     try:
055:         with NamedTemporaryFile(
056:             suffix=suffix, delete=False
057:         ) as temp:
058:             temp.write(data)
059:             temp_path = Path(temp.name)
060: 
061:         text = extract_text(temp_path)
062:         chunks = chunk_text(text, chunk_size=800, overlap=120)
063: 
064:         if not chunks:
065:             raise HTTPException(
066:                 status_code=422,
067:                 detail="No readable text found in the document."
068:             )
069: 
070:         stor

In [56]:

%pip install -q -r /content/Task-3-Advanced-RAG-Assistant/requirements.txt


In [57]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

for filename in [
    "api/main.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/ui.py",
    "Dockerfile",
]:
    path = PROJECT / filename
    print(f"{filename}: {'OK' if path.exists() else 'MISSING'}")


api/main.py: OK
api/ingestion.py: OK
api/vector_store.py: OK
api/qa.py: OK
api/ui.py: OK
Dockerfile: OK


In [58]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

for filename in [
    "api/main.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
]:
    path = PROJECT / filename
    print("\n" + "=" * 55)
    print("FILE:", filename)
    print("=" * 55)

    lines = path.read_text(encoding="utf-8").splitlines()
    print("\n".join(
        f"{i}: {line}"
        for i, line in enumerate(lines, start=1)
    ))



FILE: api/main.py
1: 
2: from pathlib import Path
3: from tempfile import NamedTemporaryFile
4: from threading import Lock
5: 
6: from fastapi import FastAPI, UploadFile, File, HTTPException
7: from pydantic import BaseModel, Field
8: 
9: from api.ingestion import extract_text, chunk_text
10: from api.vector_store import VectorStore
11: from api.qa import GroundedQA
12: 
13: app = FastAPI(
14:     title="Advanced RAG Assistant API",
15:     version="1.0.0",
16:     description="Document-grounded question answering."
17: )
18: 
19: store = VectorStore()
20: qa_model = None
21: document_loaded = False
22: qa_lock = Lock()
23: 
24: class QuestionRequest(BaseModel):
25:     question: str = Field(min_length=1, max_length=1000)
26: 
27: @app.get("/")
28: def health():
29:     return {
30:         "status": "running",
31:         "service": "Advanced RAG Assistant"
32:     }
33: 
34: @app.post("/upload")
35: async def upload_document(file: UploadFile = File(...)):
36:     global document_loa

In [59]:

%pip install -q -r /content/Task-3-Advanced-RAG-Assistant/requirements.txt


In [60]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
path = PROJECT / "api/main.py"

lines = path.read_text(encoding="utf-8").splitlines()

print("Total lines:", len(lines))

for number, line in enumerate(lines[40:], start=41):
    print(f"{number:03}: {line}")


Total lines: 118
041:             status_code=400,
042:             detail="Only PDF, TXT and Markdown files are supported."
043:         )
044: 
045:     data = await file.read()
046: 
047:     if not data:
048:         raise HTTPException(status_code=400, detail="Uploaded file is empty.")
049: 
050:     if len(data) > 20 * 1024 * 1024:
051:         raise HTTPException(status_code=413, detail="Maximum file size is 20 MB.")
052: 
053:     temp_path = None
054:     try:
055:         with NamedTemporaryFile(
056:             suffix=suffix, delete=False
057:         ) as temp:
058:             temp.write(data)
059:             temp_path = Path(temp.name)
060: 
061:         text = extract_text(temp_path)
062:         chunks = chunk_text(text, chunk_size=800, overlap=120)
063: 
064:         if not chunks:
065:             raise HTTPException(
066:                 status_code=422,
067:                 detail="No readable text found in the document."
068:             )
069: 
070:         stor

In [61]:

%pip install -q -r /content/Task-3-Advanced-RAG-Assistant/requirements.txt


In [62]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

for filename in [
    "api/main.py",
    "api/ingestion.py",
    "api/vector_store.py",
    "api/qa.py",
    "api/ui.py",
    "Dockerfile",
]:
    path = PROJECT / filename
    print(f"{filename}: {'OK' if path.exists() else 'MISSING'}")


api/main.py: OK
api/ingestion.py: OK
api/vector_store.py: OK
api/qa.py: OK
api/ui.py: OK
Dockerfile: OK


In [63]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

print("Project exists:", PROJECT.exists())
print("\n--- API main.py ---")
print((PROJECT / "api/main.py").read_text())


Project exists: True

--- API main.py ---

from pathlib import Path
from tempfile import NamedTemporaryFile
from threading import Lock

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field

from api.ingestion import extract_text, chunk_text
from api.vector_store import VectorStore
from api.qa import GroundedQA

app = FastAPI(
    title="Advanced RAG Assistant API",
    version="1.0.0",
    description="Document-grounded question answering."
)

store = VectorStore()
qa_model = None
document_loaded = False
qa_lock = Lock()

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=1000)

@app.get("/")
def health():
    return {
        "status": "running",
        "service": "Advanced RAG Assistant"
    }

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    global document_loaded

    suffix = Path(file.filename or "").suffix.lower()
    if suffix not in {".pdf", ".txt", ".md"}:
       

In [64]:

%cd /content/Task-3-Advanced-RAG-Assistant
!python -m uvicorn api.main:app --host 0.0.0.0 --port 8000


/content/Task-3-Advanced-RAG-Assistant
Loading weights: 100% 103/103 [00:00<00:00, 15300.09it/s]
INFO:     Started server process [12641]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [65]:

import requests

try:
    r = requests.get("http://127.0.0.1:8000/", timeout=5)
    print("Status:", r.status_code)
    print("Response:", r.text[:1000])
except Exception as e:
    print("Server check:", repr(e))


Status: 200
Response: {"status":"running","service":"Advanced RAG Assistant"}


In [66]:

import requests

for endpoint in ["/", "/openapi.json"]:
    try:
        r = requests.get(
            "http://127.0.0.1:8000" + endpoint,
            timeout=10
        )
        print("\nEndpoint:", endpoint)
        print("Status:", r.status_code)
        print(r.text[:1200])
    except Exception as e:
        print(endpoint, "ERROR:", repr(e))



Endpoint: /
Status: 200
{"status":"running","service":"Advanced RAG Assistant"}

Endpoint: /openapi.json
Status: 200
{"openapi":"3.1.0","info":{"title":"Advanced RAG Assistant API","description":"Document-grounded question answering.","version":"1.0.0"},"paths":{"/":{"get":{"summary":"Health","operationId":"health__get","responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}}}}},"/upload":{"post":{"summary":"Upload Document","operationId":"upload_document_upload_post","requestBody":{"content":{"multipart/form-data":{"schema":{"$ref":"#/components/schemas/Body_upload_document_upload_post"}}},"required":true},"responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}},"422":{"description":"Validation Error","content":{"application/json":{"schema":{"$ref":"#/components/schemas/HTTPValidationError"}}}}}}},"/ask":{"post":{"summary":"Ask Question","operationId":"ask_question_ask_post","requestBody":{"cont

In [67]:

import requests

data = requests.get(
    "http://127.0.0.1:8000/openapi.json",
    timeout=10
).json()

print("API:", data["info"]["title"])
print("\nAvailable endpoints:")

for path, methods in data["paths"].items():
    for method, details in methods.items():
        print(method.upper(), path, "-", details.get("summary", ""))


API: Advanced RAG Assistant API

Available endpoints:
GET / - Health
POST /upload - Upload Document
POST /ask - Ask Question


In [68]:

from IPython.display import display, Markdown

display(Markdown(
    "API documentation: "
    "http://127.0.0.1:8000/docs"
))


API documentation: http://127.0.0.1:8000/docs

In [69]:

import requests

url = "http://127.0.0.1:8000/openapi.json"
response = requests.get(url, timeout=10)

print("Status:", response.status_code)

if response.status_code == 200:
    api_data = response.json()

    print("API:", api_data["info"]["title"])
    print("\nAvailable endpoints:")

    for path, methods in api_data["paths"].items():
        for method, details in methods.items():
            print(
                method.upper(),
                path,
                "-",
                details.get("summary", "")
            )
else:
    print(response.text[:1500])


Status: 200
API: Advanced RAG Assistant API

Available endpoints:
GET / - Health
POST /upload - Upload Document
POST /ask - Ask Question


In [70]:

import requests
import json

url = "http://127.0.0.1:8000/ask"

payload = {
    "question": "What is the example chunk size?"
}

response = requests.post(
    url,
    json=payload,
    timeout=120
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))


Status: 200
{
  "question": "What is the example chunk size?",
  "answer": "800 characters",
  "sources": [
    {
      "text": "in the documents.\n5. Example Facts for Testing\nThe fictional test project is named PaperPilot. It uses a chunk size of 800 characters and an overlap of 120\ncharacters for its initial experiment. The embedding model used in this example is all-MiniLM-L6-v2. These\nsettings are illustrative, not universal best practices.\nSuggested test questions\n\u007f What is machine learning? \u007f What does an embedding represent? \u007f Why does the system split text into chunks? \u007f\nWhat is the fictional project called? \u007f Which chunk size is used in the example? \u007f What should the system do if the\nanswer is not in the document?",
      "score": 0.48562806844711304
    },
    {
      "text": "sed learning, unsupervised learning, and\nreinforcement learning.\n3. Embeddings and Vector Search\nAn embedding represents text as a list of numbers called a vecto

In [71]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
ui_file = PROJECT / "api" / "ui.py"

print("UI file exists:", ui_file.exists())

if ui_file.exists():
    print(ui_file.read_text()[:10000])


UI file exists: True

import streamlit as st
import requests

st.set_page_config(
    page_title="Advanced RAG Assistant",
    page_icon="📚",
    layout="wide"
)

st.title("Advanced RAG Document Assistant")
st.caption("Upload a document and ask questions grounded in its content.")

API_URL = "http://127.0.0.1:8000"

uploaded_file = st.file_uploader(
    "Upload a document",
    type=["pdf", "txt", "md"]
)

if uploaded_file is not None:
    if st.button("Upload and index document"):
        try:
            response = requests.post(
                f"{API_URL}/upload",
                files={
                    "file": (
                        uploaded_file.name,
                        uploaded_file.getvalue(),
                        uploaded_file.type or "application/octet-stream"
                    )
                },
                timeout=180
            )
            response.raise_for_status()
            st.session_state["document_ready"] = True
            st.success("Doc

In [72]:

import importlib.util

for package in ["streamlit", "requests"]:
    print(
        package,
        "INSTALLED" if importlib.util.find_spec(package) else "MISSING"
    )


streamlit INSTALLED
requests INSTALLED


In [73]:

%cd /content/Task-3-Advanced-RAG-Assistant
!streamlit run api/ui.py --server.port 8501 --server.address 0.0.0.0


/content/Task-3-Advanced-RAG-Assistant


2026-10-23 04:38:47.605 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.14.37:8501

  Stopping...
  Stopping...


In [74]:

import requests

for port in [8000, 8501]:
    try:
        r = requests.get(
            f"http://127.0.0.1:{port}/",
            timeout=5
        )
        print(f"Port {port}: HTTP {r.status_code}")
        print(r.text[:200])
    except Exception as e:
        print(f"Port {port}: ERROR - {e}")


Port 8000: ERROR - HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b9547906470>: Failed to establish a new connection: [Errno 111] Connection refused'))
Port 8501: ERROR - HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b9547bd9480>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [75]:

%cd /content/Task-3-Advanced-RAG-Assistant
!python -m uvicorn api.main:app --host 127.0.0.1 --port 8000


/content/Task-3-Advanced-RAG-Assistant
Loading weights: 100% 103/103 [00:00<00:00, 5510.80it/s]
INFO:     Started server process [16307]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [16307]
^C


In [77]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

print("Project folder:", PROJECT.exists())
print("Main API:", (PROJECT / "api" / "main.py").exists())
print("UI file:", (PROJECT / "api" / "ui.py").exists())
print("Requirements:", (PROJECT / "requirements.txt").exists())


Project folder: True
Main API: True
UI file: True
Requirements: True


In [78]:

!pip -q install gradio


In [79]:

import gradio as gr
from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

def check_project():
    if PROJECT.exists():
        return (
            "Project folder found ✅\n"
            f"Main API: {(PROJECT / 'api' / 'main.py').exists()}\n"
            f"UI file: {(PROJECT / 'api' / 'ui.py').exists()}\n"
            "Interface is ready."
        )
    return "Project folder not found. Please check the path."

with gr.Blocks(title="Advanced RAG Assistant") as demo:
    gr.Markdown("# Advanced RAG Assistant")
    gr.Markdown("Your simple project dashboard")

    status = gr.Textbox(
        label="Project Status",
        value="Click the button to check your project.",
        lines=4,
        interactive=False
    )

    check_btn = gr.Button("Check Project")
    check_btn.click(fn=check_project, outputs=status)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c996f0cb2823dbefd3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [80]:

from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

for name in ["api/main.py", "api/ingestion.py",
             "api/vector_store.py", "api/qa.py"]:
    path = PROJECT / name
    print(f"\n{'FOUND' if path.exists() else 'MISSING'}: {name}")
    if path.exists():
        print(path.read_text()[:500])



FOUND: api/main.py

from pathlib import Path
from tempfile import NamedTemporaryFile
from threading import Lock

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field

from api.ingestion import extract_text, chunk_text
from api.vector_store import VectorStore
from api.qa import GroundedQA

app = FastAPI(
    title="Advanced RAG Assistant API",
    version="1.0.0",
    description="Document-grounded question answering."
)

store = VectorStore()
qa_model = None
document_l

FOUND: api/ingestion.py

from pathlib import Path
from pypdf import PdfReader

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md"}
MAX_FILE_SIZE_MB = 20

def extract_text(file_path):
    path = Path(file_path)

    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ValueError("Only PDF, TXT and Markdown files are supported.")

    if path.stat().st_size > MA

In [81]:

from pathlib import Path

main_file = Path(
    "/content/Task-3-Advanced-RAG-Assistant/api/main.py"
)

lines = main_file.read_text().splitlines()

print("API ROUTES AND FUNCTIONS")
for i, line in enumerate(lines):
    if any(x in line for x in [
        "@app.get", "@app.post", "@app.put",
        "@app.delete", "async def ", "def "
    ]):
        print(f"\nLine {i+1}:")
        print("\n".join(
            f"{j+1}: {lines[j]}"
            for j in range(max(0, i-1), min(len(lines), i+3))
        ))


API ROUTES AND FUNCTIONS

Line 27:
26: 
27: @app.get("/")
28: def health():
29:     return {

Line 28:
27: @app.get("/")
28: def health():
29:     return {
30:         "status": "running",

Line 34:
33: 
34: @app.post("/upload")
35: async def upload_document(file: UploadFile = File(...)):
36:     global document_loaded

Line 35:
34: @app.post("/upload")
35: async def upload_document(file: UploadFile = File(...)):
36:     global document_loaded
37: 

Line 89:
88: 
89: @app.post("/ask")
90: def ask_question(request: QuestionRequest):
91:     global qa_model

Line 90:
89: @app.post("/ask")
90: def ask_question(request: QuestionRequest):
91:     global qa_model
92: 


In [82]:

from pathlib import Path

p = Path("/content/Task-3-Advanced-RAG-Assistant/api/main.py")
lines = p.read_text().splitlines()

print("Total lines:", len(lines))
print("\n--- API ROUTES ---")

for n, line in enumerate(lines, start=1):
    if "@app." in line or "@router." in line:
        print(f"{n}: {line.strip()}")


Total lines: 118

--- API ROUTES ---
27: @app.get("/")
34: @app.post("/upload")
89: @app.post("/ask")


In [83]:

import requests

url = "http://127.0.0.1:8000/"

try:
    response = requests.get(url, timeout=10)
    print("Status code:", response.status_code)
    print("Response:", response.text[:1000])
except requests.exceptions.RequestException as e:
    print("Connection error:", e)


Connection error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b9546142cf0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [84]:

%pip install -q gradio pypdf


In [85]:

import gradio as gr
from pathlib import Path
from pypdf import PdfReader
import re

def read_document(file_path):
    path = Path(file_path)
    ext = path.suffix.lower()

    if ext == ".pdf":
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    if ext in (".txt", ".md"):
        return path.read_text(encoding="utf-8", errors="ignore")

    return ""

def answer_question(files, question):
    if not files:
        return "Please upload a PDF, TXT, or Markdown document."

    if not question.strip():
        return "Please enter a question."

    results = []

    for file in files:
        text = read_document(file.name if hasattr(file, "name") else file)
        paragraphs = [
            p.strip() for p in re.split(r"\n+", text)
            if p.strip()
        ]

        terms = set(re.findall(r"\w+", question.lower()))
        scored = []

        for paragraph in paragraphs:
            words = set(re.findall(r"\w+", paragraph.lower()))
            score = len(terms & words)

            if score:
                scored.append((score, paragraph))

        scored.sort(key=lambda item: item[0], reverse=True)

        for score, paragraph in scored[:3]:
            results.append(
                f"Document: {Path(file.name if hasattr(file, 'name') else file).name}\n"
                f"Match score: {score}\n{paragraph[:1200]}"
            )

    if not results:
        return (
            "No matching text was found in the uploaded documents. "
            "Try different keywords or upload a text-based PDF."
        )

    return "\n\n---\n\n".join(results[:8])

with gr.Blocks(title="Advanced RAG Assistant Lite") as demo:
    gr.Markdown("# Advanced RAG Assistant")
    gr.Markdown(
        "### Lite Edition\n"
        "Upload a document and search for relevant information."
    )

    files = gr.File(
        label="Upload documents",
        file_count="multiple",
        file_types=[".pdf", ".txt", ".md"],
        type="filepath"
    )

    question = gr.Textbox(
        label="Your question",
        placeholder="Example: What is the example chunk size?"
    )

    submit = gr.Button("Find Answer", variant="primary")
    output = gr.Textbox(
        label="Search Results",
        lines=12
    )

    submit.click(
        fn=answer_question,
        inputs=[files, question],
        outputs=output
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7d9ca7e009bb05d5f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [86]:
from pathlib import Path

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")

print("Project folder exists:", PROJECT.exists())

for p in sorted(PROJECT.rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJECT))

Project folder exists: True
.env.example
.gitignore
.gradio/certificate.pem
Dockerfile
README.md
api/__init__.py
api/__pycache__/__init__.cpython-313.pyc
api/__pycache__/ingestion.cpython-313.pyc
api/__pycache__/main.cpython-313.pyc
api/__pycache__/qa.cpython-313.pyc
api/__pycache__/rag.cpython-313.pyc
api/__pycache__/ui.cpython-313.pyc
api/__pycache__/vector_store.cpython-313.pyc
api/ingestion.py
api/main.py
api/qa.py
api/rag.py
api/ui.py
api/vector_store.py
requirements.txt
sample_documents/sample.txt


In [87]:
from pathlib import Path
import shutil

PROJECT = Path("/content/Task-3-Advanced-RAG-Assistant")
ZIP_PATH = "/content/Task-3-Advanced-RAG-Assistant"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    root_dir="/content",
    base_dir="Task-3-Advanced-RAG-Assistant"
)

print("ZIP created:", ZIP_PATH + ".zip")

ZIP created: /content/Task-3-Advanced-RAG-Assistant.zip
